# Week 13: The Reference Customer Effect — Causal Network Analysis
## Network-Adjusted CLV & Instrumental Variables

**Masterclass:** [Week 13 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week13_causal_network.html)

**Dataset:** Simulated customer network (generated in-notebook)

**What You'll Build:**
1. Customer network graph construction
2. Peer influence variables (neighbor churn rates)
3. Naive vs. IV-corrected peer effect estimation
4. Network-adjusted CLV calculation

In [ ]:
!pip install networkx pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Generate Customer Network

We simulate a realistic customer network where: (a) customers are connected via referrals, shared accounts, or social ties, and (b) churn has both individual drivers and peer influence.

In [ ]:
np.random.seed(42)
n_customers = 2000
n_edges = 4000

# Generate customer features
customers = pd.DataFrame({
    'customer_id': range(n_customers),
    'tenure': np.random.exponential(24, n_customers).clip(1, 72).astype(int),
    'monthly_charges': np.random.normal(70, 20, n_customers).clip(20, 120),
    'contract_months': np.random.choice([0, 12, 24], n_customers, p=[0.5, 0.3, 0.2]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_customers),
})

# IV: contract expiry timing (exogenous shock to peer's churn probability)
customers['months_to_renewal'] = np.where(
    customers['contract_months'] > 0,
    np.random.randint(0, 12, n_customers),
    0
)

# Generate network edges (preferential attachment-like)
edges = set()
while len(edges) < n_edges:
    a, b = np.random.choice(n_customers, 2, replace=False)
    if a != b:
        edges.add((min(a,b), max(a,b)))
edges = list(edges)

G = nx.Graph()
G.add_nodes_from(range(n_customers))
G.add_edges_from(edges)

print(f"Customers: {n_customers:,}")
print(f"Connections: {len(edges):,}")
print(f"Avg degree: {np.mean([d for _, d in G.degree()]):.1f}")

In [ ]:
# Simulate churn with peer influence
# True model: P(churn) = logistic(base + individual + 0.3 * peer_churn_rate)
base_logit = -1.5
beta_charges = 0.015
beta_contract = -0.8
TRUE_PEER_EFFECT = 0.3

# Step 1: generate initial churn propensity (without peer effects)
customers['churn_propensity'] = (base_logit
    + beta_charges * (customers['monthly_charges'] - 70)
    + beta_contract * (customers['contract_months'] > 0).astype(float)
    + 0.5 * (customers['months_to_renewal'] < 2).astype(float))

# Step 2: iterative peer influence (2 rounds of contagion)
customers['churned'] = (np.random.rand(n_customers) <
    1 / (1 + np.exp(-customers['churn_propensity']))).astype(int)

for _ in range(2):
    # Compute peer churn rate
    peer_churn = []
    for i in range(n_customers):
        neighbors = list(G.neighbors(i))
        if neighbors:
            peer_churn.append(customers.loc[neighbors, 'churned'].mean())
        else:
            peer_churn.append(0)
    customers['peer_churn_rate'] = peer_churn

    # Update churn with peer influence
    updated_logit = customers['churn_propensity'] + TRUE_PEER_EFFECT * customers['peer_churn_rate']
    customers['churned'] = (np.random.rand(n_customers) <
        1 / (1 + np.exp(-updated_logit))).astype(int)

# Final peer churn rate
peer_churn = []
for i in range(n_customers):
    neighbors = list(G.neighbors(i))
    peer_churn.append(customers.loc[neighbors, 'churned'].mean() if neighbors else 0)
customers['peer_churn_rate'] = peer_churn

print(f"Overall churn rate: {customers['churned'].mean():.1%}")
print(f"Churn rate where peer_churn > 50%: {customers.loc[customers['peer_churn_rate']>0.5, 'churned'].mean():.1%}")
print(f"Churn rate where peer_churn < 20%: {customers.loc[customers['peer_churn_rate']<0.2, 'churned'].mean():.1%}")

---
## Part 2: Naive vs. Causal Peer Effect

**Manski's reflection problem:** Naive regression conflates contagion (true peer effect) with homophily (similar people cluster). The naive estimate will be biased upward.

In [ ]:
import statsmodels.api as sm

# Naive OLS
X_naive = sm.add_constant(customers[['peer_churn_rate', 'monthly_charges',
                                      'contract_months', 'tenure']])
naive_model = sm.OLS(customers['churned'], X_naive).fit(cov_type='HC1')
print("=== Naive OLS (biased) ===")
print(f"Peer churn effect: {naive_model.params['peer_churn_rate']:.3f} (p={naive_model.pvalues['peer_churn_rate']:.4f})")
print(f"True effect: {TRUE_PEER_EFFECT:.3f}")
print(f"Bias: {naive_model.params['peer_churn_rate'] - TRUE_PEER_EFFECT:+.3f} (overestimates by {((naive_model.params['peer_churn_rate']/TRUE_PEER_EFFECT)-1)*100:.0f}%)")

In [ ]:
# IV approach: use peer's months_to_renewal as instrument
# Peers near contract expiry are exogenously more likely to churn
# This shock is unrelated to YOUR characteristics (exclusion restriction)

peer_iv = []
for i in range(n_customers):
    neighbors = list(G.neighbors(i))
    if neighbors:
        peer_iv.append(customers.loc[neighbors, 'months_to_renewal'].mean())
    else:
        peer_iv.append(6)  # neutral value
customers['peer_renewal_iv'] = peer_iv

# 2SLS: Stage 1 - predict peer churn from instrument
X_stage1 = sm.add_constant(customers[['peer_renewal_iv', 'monthly_charges',
                                       'contract_months', 'tenure']])
stage1 = sm.OLS(customers['peer_churn_rate'], X_stage1).fit()
customers['peer_churn_predicted'] = stage1.fittedvalues

print("=== Stage 1: First-stage F-statistic ===")
# F-stat for instrument strength
f_stat = stage1.fvalue
print(f"F-statistic: {f_stat:.1f} ({'Strong' if f_stat > 10 else 'Weak'} instrument)")

# Stage 2 - regress churn on predicted peer churn
X_stage2 = sm.add_constant(customers[['peer_churn_predicted', 'monthly_charges',
                                       'contract_months', 'tenure']])
stage2 = sm.OLS(customers['churned'], X_stage2).fit(cov_type='HC1')

print(f"\n=== IV-Corrected Peer Effect (2SLS) ===")
print(f"Estimated: {stage2.params['peer_churn_predicted']:.3f}")
print(f"True: {TRUE_PEER_EFFECT:.3f}")
print(f"Bias: {stage2.params['peer_churn_predicted'] - TRUE_PEER_EFFECT:+.3f}")

---
## Part 3: Network-Adjusted CLV

In [ ]:
# Standard CLV (direct value only)
customers['direct_clv'] = customers['monthly_charges'] * customers['tenure'] * 0.4

# Network CLV: direct + influence value
PEER_EFFECT = stage2.params['peer_churn_predicted']
customers['degree'] = [G.degree(i) for i in range(n_customers)]

# Influence value: if I churn, my neighbors lose PEER_EFFECT * their CLV probability
customers['influence_value'] = 0.0
for i in range(n_customers):
    neighbors = list(G.neighbors(i))
    if neighbors:
        neighbor_clv = customers.loc[neighbors, 'direct_clv'].sum()
        customers.loc[i, 'influence_value'] = abs(PEER_EFFECT) * neighbor_clv

customers['network_clv'] = customers['direct_clv'] + customers['influence_value']

# Top influencers
top = customers.nlargest(10, 'network_clv')
print("=== Top 10 by Network-Adjusted CLV ===")
print(top[['customer_id', 'direct_clv', 'influence_value', 'network_clv', 'degree']].to_string(index=False))

multiplier = customers['network_clv'].sum() / customers['direct_clv'].sum()
print(f"\nNetwork CLV multiplier: {multiplier:.2f}x")
print(f"Customers where influence_value > direct_clv: {(customers['influence_value'] > customers['direct_clv']).sum():,}")

---
## Exercises

### Exercise 1: Visualize the Network
Plot a subgraph of the top 50 most connected customers, colored by churn status. Do churned clusters emerge?

### Exercise 2: Contagion Simulation
If you prevent the top 10 network-CLV customers from churning, how much does the overall churn rate decrease? Simulate it.

### Exercise 3: Weak Instruments
What happens if you use a weak instrument (F < 10)? Try using `region` as the instrument instead. How does the estimate change?

---
## Key Takeaways
1. **Naive peer effect estimates are biased upward** due to homophily
2. **IV (2SLS)** purges the bias using exogenous shocks to peer behavior
3. **Network-adjusted CLV** can be 2-20x direct CLV for high-influence customers
4. **Retaining reference customers** has outsized impact on the entire network

---
**Congratulations!** You've completed the Q1 foundation: 13 weeks from survival curves through causal network effects. Q2 begins with pricing optimization.